In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

class SimplifiedAttentionClassifier:
    def __init__(self, max_nodes=10, node_dim=10):
        """
        简化版注意力分类器 - 专用于加载预训练模型进行推理
        注意：必须与训练时的特征提取完全一致！
        """
        self.max_nodes = max_nodes
        self.node_dim = node_dim  # 必须是10
        self.graph_data = []
        self.labels = []
        self.model = None
        
        # 那非化合物的特征峰（必须与训练时完全一致）
        self.nonafi_characteristic_peaks = [
            58.0651, 72.0808, 84.0808, 99.0917, 113.1073,
            135.0441, 147.0077, 151.0866, 166.0975, 169.076,
            197.0709, 250.0863, 256.0955, 262.0862, 283.1195,
            297.1346, 299.1139, 302.0812, 312.1581, 315.091,
            327.1274, 341.1608, 354.2, 377.1, 396.203
        ]
        
        # 峰组分类（必须与训练时完全一致）
        self.peak_groups = {
            'low_mass': [58.0651, 72.0808, 84.0808, 99.0917, 113.1073],
            'middle_mass': [135.0441, 147.0077, 151.0866, 166.0975, 169.076, 197.0709],
            'high_mass': [250.0863, 256.0955, 262.0862, 283.1195, 297.1346, 299.1139, 302.0812],
            'very_high_mass': [312.1581, 315.091, 327.1274, 341.1608, 354.2, 377.1, 396.203]
        }
        
        # 关键特征峰（必须与训练时完全一致）
        self.key_peaks = [58.0651, 72.0808, 135.0441, 166.0975, 250.0863]
        
        print(f"简化版注意力分类器初始化完成")
        print(f"节点特征维度: {node_dim} (必须与训练时一致)")
        print(f"特征列表: 10个特征，无intensity相关特征")
    
    def load_data_for_prediction(self, file_path, ms_column='peaks'):
        """
        仅加载数据用于预测（不需要训练）
        参数:
            file_path: 文件路径
            ms_column: 质谱数据列名，默认为'peaks'
        """
        print(f"正在加载数据用于预测: {file_path}")
        print(f"使用质谱数据列: '{ms_column}'")
        
        # 根据文件扩展名读取数据
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path, encoding='utf-8')
        elif file_path.endswith('.xlsx'):
            df = pd.read_excel(file_path)
        else:
            raise ValueError("仅支持.csv和.xlsx格式的文件")
        
        # 检查peaks列是否存在
        if ms_column not in df.columns:
            print(f"❌ 错误: 列 '{ms_column}' 不存在")
            print(f"可用的列名: {list(df.columns)}")
            # 尝试查找可能的质谱数据列
            possible_cols = []
            for col in df.columns:
                sample_val = str(df[col].iloc[0]) if len(df) > 0 else ""
                if ':' in sample_val and any(c.isdigit() for c in sample_val):
                    possible_cols.append(col)
            
            if possible_cols:
                print(f"\n📊 发现可能的质谱数据列: {possible_cols}")
                print("请修改代码中的ms_column参数")
            
            return None
        
        print(f"✅ 成功找到质谱数据列: '{ms_column}'")
        print(f"数据形状: {df.shape}")
        print(f"前3行预览:")
        for i in range(min(3, len(df))):
            print(f"  行{i}: {str(df[ms_column].iloc[i])[:80]}...")
        
        # 预处理标签（如果存在）
        if 'label' in df.columns:
            labels_cleaned = []
            for val in df['label']:
                if pd.isna(val):
                    labels_cleaned.append(0)
                else:
                    str_val = str(val).strip().lower()
                    if str_val in ['0', '0.0', '非那非', '否', 'negative', 'n', 'false', 'f', '非']:
                        labels_cleaned.append(0)
                    elif str_val in ['1', '1.0', '那非', '是', 'positive', 'y', 'true', 't', '是']:
                        labels_cleaned.append(1)
                    else:
                        try:
                            num_val = float(str_val)
                            labels_cleaned.append(int(num_val) if num_val in [0, 1] else 0)
                        except:
                            labels_cleaned.append(0)
            self.labels = np.array(labels_cleaned, dtype=int)
            print(f"已加载 {len(self.labels)} 个样本标签")
            print(f"标签分布 - 那非(1): {sum(self.labels)}, 非那非(0): {len(self.labels)-sum(self.labels)}")
        else:
            print("警告: 数据中没有标签列，将仅进行预测")
            self.labels = np.zeros(len(df), dtype=int)
        
        # 计算统计信息（必须与训练时相同逻辑）
        all_mz = []
        all_max_intensity_mz = []
        
        for i, row in df.iterrows():
            ms_str = str(row[ms_column]) if not pd.isna(row[ms_column]) else ""
            
            if pd.isna(ms_str) or ms_str == 'nan' or ms_str.strip() == '':
                continue
                
            # 解析峰数据
            peaks = ms_str.replace(';', ',').split(',')
            max_intensity = -1
            max_intensity_mz = 0
            
            for peak in peaks[:self.max_nodes]:
                try:
                    peak = peak.strip()
                    if not peak:
                        continue
                    
                    parts = peak.split(':')
                    if len(parts) >= 2:
                        mz = float(parts[0].strip())
                        intensity = float(parts[1].strip())
                        all_mz.append(mz)
                        
                        # 记录最大强度对应的m/z
                        if intensity > max_intensity:
                            max_intensity = intensity
                            max_intensity_mz = mz
                    elif len(parts) == 1:
                        mz = float(parts[0].strip())
                        intensity = 1.0
                        all_mz.append(mz)
                        
                        if intensity > max_intensity:
                            max_intensity = intensity
                            max_intensity_mz = mz
                except:
                    continue
            
            if max_intensity > 0:
                all_max_intensity_mz.append(max_intensity_mz)
        
        # 计算统计量（必须与训练时相同）
        if all_mz:
            self.mz_mean = np.mean(all_mz)
            self.mz_std = np.std(all_mz) if np.std(all_mz) > 0 else 1
        else:
            self.mz_mean = 0
            self.mz_std = 1
        
        if all_max_intensity_mz:
            self.max_intensity_mz_mean = np.mean(all_max_intensity_mz)
            self.max_intensity_mz_std = np.std(all_max_intensity_mz) if np.std(all_max_intensity_mz) > 0 else 1
        else:
            self.max_intensity_mz_mean = 0
            self.max_intensity_mz_std = 1
        
        print(f"\n统计信息:")
        print(f"m/z均值: {self.mz_mean:.2f}, 标准差: {self.mz_std:.2f}")
        print(f"最大强度m/z均值: {self.max_intensity_mz_mean:.2f}, 标准差: {self.max_intensity_mz_std:.2f}")
        
        # 构建图数据
        print("\n构建图表示...")
        self.graph_data = []  # 清空之前的数据
        
        for i, row in df.iterrows():
            ms_str = str(row[ms_column]) if not pd.isna(row[ms_column]) else ""
            
            if ms_str is None or pd.isna(ms_str) or ms_str == 'nan' or ms_str.strip() == '':
                node_features = np.zeros((self.max_nodes, self.node_dim))
                adjacency_matrix = np.eye(self.max_nodes)
            else:
                node_features, adjacency_matrix = self._parse_ms_string(ms_str)
            
            self.graph_data.append({
                'node_features': node_features,
                'adjacency_matrix': adjacency_matrix
            })
        
        print(f"数据处理完成! 构建了 {len(self.graph_data)} 个图")
        return df
    
    def _parse_ms_string(self, ms_str):
        """解析质谱字符串为图数据（必须与训练时相同逻辑）"""
        peaks = ms_str.replace(';', ',').split(',')
        peak_data = []
        
        # 首先找到最大强度的m/z
        max_intensity = -1
        max_intensity_mz = 0
        
        for peak in peaks:
            try:
                peak = peak.strip()
                if not peak:
                    continue
                
                parts = peak.split(':')
                if len(parts) >= 2:
                    mz = float(parts[0].strip())
                    intensity = float(parts[1].strip())
                    
                    # 记录最大强度
                    if intensity > max_intensity:
                        max_intensity = intensity
                        max_intensity_mz = mz
                    
                    peak_data.append((mz, intensity))
                elif len(parts) == 1:
                    mz = float(parts[0].strip())
                    intensity = 1.0
                    
                    if intensity > max_intensity:
                        max_intensity = intensity
                        max_intensity_mz = mz
                    
                    peak_data.append((mz, intensity))
            except:
                continue
        
        # 按强度降序排序（必须与训练时相同）
        peak_data.sort(key=lambda x: x[1], reverse=True)
        peak_data = peak_data[:self.max_nodes]
        
        # 提取m/z值
        mz_values = []
        for j, (mz, intensity) in enumerate(peak_data):
            mz_values.append(mz)
        
        # 计算节点特征（必须与训练时相同）
        node_features = np.zeros((self.max_nodes, self.node_dim))
        for j in range(self.max_nodes):
            if j < len(peak_data):
                mz = peak_data[j][0]
                intensity = peak_data[j][1]
            elif len(peak_data) > 0:
                mz = peak_data[-1][0]
                intensity = 1.0
            else:
                mz = 0
                intensity = 0
            
            features = self._compute_node_features(mz, j, len(peak_data), mz_values, max_intensity_mz)
            for k in range(min(len(features), self.node_dim)):
                node_features[j, k] = features[k]
        
        # 构建邻接矩阵（必须与训练时相同）
        adjacency_matrix = self._build_adjacency_matrix(mz_values)
        
        return node_features, adjacency_matrix
    
    def _compute_node_features(self, mz, position, total_peaks, all_mz_values, max_intensity_mz):
        """计算节点特征（10维，必须与训练时完全相同）"""
        # 1. 标准化m/z
        mz_norm = (mz - self.mz_mean) / self.mz_std
        
        # 2. 位置特征（按intensity排序）
        position_ratio = position / max(total_peaks, 1)
        is_first_peak = 1.0 if position == 0 else 0.0  # 强度最大的峰
        is_last_peak = 1.0 if position == total_peaks - 1 else 0.0  # 强度最小的峰
        
        # 3. 那非特征峰匹配（只到小数点后1位）
        rounded_mz = round(mz, 1)
        rounded_characteristic = [round(p, 1) for p in self.nonafi_characteristic_peaks]
        is_characteristic = 1.0 if rounded_mz in rounded_characteristic else 0.0
        
        # 4. 与最近特征峰的m/z差异
        if self.nonafi_characteristic_peaks:
            min_diff = min([abs(mz - cp) for cp in self.nonafi_characteristic_peaks])
            characteristic_mz_diff = min_diff / 100.0  # 标准化差异
        else:
            characteristic_mz_diff = 1.0
        
        # 5. 质量区域特征（匹配到小数点后1位）
        mass_region_feature = 0.0
        for group_name, group_peaks in self.peak_groups.items():
            group_rounded = [round(p, 1) for p in group_peaks]
            if rounded_mz in group_rounded:
                mass_region_feature = {'low_mass': 0.25, 'middle_mass': 0.5, 
                                     'high_mass': 0.75, 'very_high_mass': 1.0}[group_name]
                break
        
        # 6. 是否关键峰（匹配到小数点后1位）
        rounded_key_peaks = [round(p, 1) for p in self.key_peaks]
        is_key_peak = 1.0 if rounded_mz in rounded_key_peaks else 0.0
        
        # 7. 最大强度m/z特征
        if max_intensity_mz > 0:
            max_intensity_mz_norm = (max_intensity_mz - self.max_intensity_mz_mean) / self.max_intensity_mz_std
            mz_relative_to_max = mz / max_intensity_mz
        else:
            max_intensity_mz_norm = 0.0
            mz_relative_to_max = 1.0
        
        # 构建特征向量（10维，必须与训练时相同顺序）
        features = [
            mz_norm,                    # 0: 标准化m/z
            position_ratio,             # 1: 位置比例（按intensity排序）
            is_first_peak,              # 2: 是否强度最大的峰
            is_last_peak,               # 3: 是否强度最小的峰
            is_characteristic,          # 4: 是否特征峰
            characteristic_mz_diff,     # 5: 与最近特征峰的m/z差异
            mass_region_feature,        # 6: 质量区域特征
            is_key_peak,                # 7: 是否关键峰
            max_intensity_mz_norm,      # 8: 最大强度m/z标准化
            mz_relative_to_max          # 9: 相对最大强度m/z的比值
        ]
        
        return features
    
    def _build_adjacency_matrix(self, mz_values):
        """构建邻接矩阵（必须与训练时相同）"""
        n_nodes = len(mz_values)
        adjacency_matrix = np.eye(self.max_nodes)
        
        if n_nodes > 0:
            for i in range(min(n_nodes, self.max_nodes)):
                for j in range(min(n_nodes, self.max_nodes)):
                    if i != j:
                        mz_diff = abs(mz_values[i] - mz_values[j])
                        similarity = np.exp(-mz_diff**2 / (2 * 50.0**2))
                        adjacency_matrix[i, j] = similarity
        
        return adjacency_matrix
    
    def prepare_batch_data(self):
        """准备批次数据"""
        batch_size = len(self.graph_data)
        nodes_batch = np.zeros((batch_size, self.max_nodes, self.node_dim))
        adj_batch = np.zeros((batch_size, self.max_nodes, self.max_nodes))
        
        for i, graph in enumerate(self.graph_data):
            nodes_batch[i] = graph['node_features']
            adj_batch[i] = graph['adjacency_matrix']
        
        return [nodes_batch, adj_batch]
    
    def load_best_model(self, model_path='grid_search_best_model_no_intensity_h8_acc_0.9750_auc_0.9859.h5'):
        """加载最佳模型"""
        print(f"正在加载最佳模型: {model_path}")
        try:
            self.model = tf.keras.models.load_model(model_path)
            print("✅ 模型加载成功!")
            
            # 验证模型输入形状
            print(f"模型输入形状: {self.model.input_shape}")
            print(f"期望节点特征形状: (None, {self.max_nodes}, {self.node_dim})")
        except Exception as e:
            print(f"❌ 模型加载失败: {e}")
            print("请确保模型文件存在且路径正确")
            self.model = None
        
        return self.model
    
    def predict_and_evaluate(self):
        """进行预测和评估"""
        if self.model is None:
            print("❌ 错误: 模型未加载")
            return None, None
        
        print("\n" + "="*60)
        print("开始预测和评估")
        print("="*60)
        
        # 准备数据
        X = self.prepare_batch_data()
        
        # 预测
        print("正在进行预测...")
        y_pred_prob = self.model.predict(X, verbose=1).flatten()
        y_pred = (y_pred_prob > 0.5).astype(int)
        
        print(f"\n预测结果统计:")
        print(f"预测概率范围: [{y_pred_prob.min():.4f}, {y_pred_prob.max():.4f}]")
        print(f"预测均值: {y_pred_prob.mean():.4f}")
        print(f"预测为那非(1)的数量: {y_pred.sum()}")
        print(f"预测为非那非(0)的数量: {len(y_pred) - y_pred.sum()}")
        print(f"那非预测率: {y_pred.sum()/len(y_pred):.2%}")
        
        # 如果有真实标签，进行评估
        if len(self.labels) > 0 and np.any(self.labels != 0) and len(self.labels) == len(y_pred):
            y_true = self.labels
            
            accuracy = accuracy_score(y_true, y_pred)
            print(f"\n评估结果:")
            print(f"准确率: {accuracy:.4f}")
            
            try:
                auc_score = roc_auc_score(y_true, y_pred_prob)
                print(f"AUC: {auc_score:.4f}")
                
                # 绘制ROC曲线
                fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
                roc_auc = auc(fpr, tpr)
                
                plt.figure(figsize=(8, 6))
                plt.plot(fpr, tpr, color='darkorange', lw=2, 
                        label=f'ROC曲线 (AUC = {roc_auc:.4f})')
                plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
                plt.xlim([0.0, 1.0])
                plt.ylim([0.0, 1.05])
                plt.xlabel('假阳性率')
                plt.ylabel('真阳性率')
                plt.title('ROC曲线 - 外部验证')
                plt.legend(loc="lower right")
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.savefig('external_validation_roc_curve.png', dpi=300)
                plt.show()
            except Exception as e:
                print(f"无法计算AUC: {e}")
            
            # 混淆矩阵
            if len(set(y_true)) > 1:
                cm = confusion_matrix(y_true, y_pred)
                self.plot_confusion_matrix(cm)
            else:
                print("⚠️  注意：所有样本的标签相同，无法绘制混淆矩阵")
            
            # 分类报告
            print("\n分类报告:")
            print(classification_report(y_true, y_pred, target_names=['非那非', '那非']))
        else:
            print("⚠️  注意：没有真实标签或标签数量不匹配，无法评估模型性能")
        
        return y_pred, y_pred_prob
    
    def plot_confusion_matrix(self, cm):
        """绘制混淆矩阵"""
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=['非那非', '那非'],
                   yticklabels=['非那非', '那非'])
        plt.title('混淆矩阵 - 外部验证')
        plt.ylabel('真实标签')
        plt.xlabel('预测标签')
        plt.tight_layout()
        plt.savefig('external_validation_confusion_matrix.png', dpi=300)
        plt.show()
    
    def save_predictions(self, original_df, y_pred, y_pred_prob, output_file='external_validation_predictions.xlsx'):
        """保存预测结果到Excel"""
        print(f"\n正在保存预测结果到: {output_file}")
        
        result_df = original_df.copy()
        result_df['预测概率'] = y_pred_prob
        result_df['预测标签'] = y_pred
        result_df['预测类别'] = result_df['预测标签'].map({0: '非那非', 1: '那非'})
        
        # 如果有真实标签，添加正确性判断
        if 'label' in result_df.columns and len(result_df) == len(y_pred):
            result_df['预测正确'] = (result_df['预测标签'] == result_df['label']).astype(int)
        
        # 保存到Excel
        result_df.to_excel(output_file, index=False)
        print(f"✅ 预测结果已保存!")
        
        # 打印统计信息
        print(f"\n预测统计信息:")
        print(f"总样本数: {len(result_df)}")
        print(f"预测为那非的数量: {result_df['预测标签'].sum()}")
        print(f"预测为非那非的数量: {len(result_df) - result_df['预测标签'].sum()}")
        
        if '预测正确' in result_df.columns and 'label' in result_df.columns:
            if result_df['label'].nunique() > 1:  # 有多类标签
                accuracy = result_df['预测正确'].mean()
                print(f"预测准确率: {accuracy:.4f}")
            else:
                print("⚠️  注意：所有样本的标签相同，无法计算准确率")
        
        # 显示前几个预测结果
        print(f"\n前5个预测结果:")
        display_cols = []
        if '预测概率' in result_df.columns:
            display_cols.append('预测概率')
        if '预测类别' in result_df.columns:
            display_cols.append('预测类别')
        if 'label' in result_df.columns:
            display_cols.append('label')
        
        print(result_df[display_cols].head())
        
        return result_df


def test_feature_extraction():
    """测试特征提取是否与训练时一致"""
    print("="*80)
    print("测试特征提取一致性")
    print("="*80)
    
    # 创建一个测试实例
    classifier = SimplifiedAttentionClassifier(max_nodes=10, node_dim=10)
    
    # 模拟训练时的统计信息（与实际训练时相同）
    classifier.mz_mean = 150.0
    classifier.mz_std = 50.0
    classifier.max_intensity_mz_mean = 200.0
    classifier.max_intensity_mz_std = 40.0
    
    # 测试一个样本
    test_mz = 135.0441  # 那非特征峰
    features = classifier._compute_node_features(
        mz=test_mz,
        position=0,
        total_peaks=5,
        all_mz_values=[135.0441, 250.0863, 58.0651],
        max_intensity_mz=250.0863
    )
    
    print(f"测试m/z: {test_mz}")
    print(f"特征数量: {len(features)}")
    print(f"特征值: {[f'{x:.4f}' for x in features]}")
    print(f"特征维度验证: {'✓ 通过' if len(features) == 10 else '✗ 失败'}")
    
    return classifier


def main_external_validation():
    """主函数：直接加载最佳模型进行外部验证"""
    print("="*80)
    print("🚀 直接加载最佳模型进行外部验证")
    print("="*80)
    
    # 1. 测试特征提取
    print("第一步：测试特征提取一致性")
    test_feature_extraction()
    
    # 2. 初始化分类器
    print("\n\n" + "="*80)
    print("第二步：初始化分类器")
    print("="*80)
    classifier = SimplifiedAttentionClassifier(max_nodes=10, node_dim=10)
    
    # 3. 加载最佳模型
    print("\n" + "="*80)
    print("第三步：加载预训练模型")
    print("="*80)
    best_model_path = "grid_search_best_model_no_intensity_h8_acc_0.9750_auc_0.9859.h5"
    classifier.load_best_model(best_model_path)
    
    if classifier.model is None:
        print("❌ 模型加载失败，程序终止")
        return
    
    # 4. 加载外部验证数据
    print("\n" + "="*80)
    print("第四步：加载外部验证数据")
    print("="*80)
    external_file_path = "外部验证1225.xlsx"  # 你的Excel文件
    
    try:
        # 使用peaks列作为质谱数据
        external_df = classifier.load_data_for_prediction(external_file_path, ms_column='peaks')
        if external_df is None:
            print("❌ 数据加载失败，程序终止")
            return
    except Exception as e:
        print(f"❌ 加载数据失败: {e}")
        import traceback
        traceback.print_exc()
        print(f"请确保文件 '{external_file_path}' 存在且格式正确")
        return
    
    # 5. 进行预测和评估
    print("\n" + "="*80)
    print("第五步：进行预测和评估")
    print("="*80)
    y_pred, y_pred_prob = classifier.predict_and_evaluate()
    
    if y_pred is None:
        print("❌ 预测失败，程序终止")
        return
    
    # 6. 保存结果
    print("\n" + "="*80)
    print("第六步：保存预测结果")
    print("="*80)
    result_df = classifier.save_predictions(
        external_df, 
        y_pred, 
        y_pred_prob,
        output_file='外部验证1225_预测结果-2.xlsx'
    )
    
    # 7. 总结
    print("\n" + "="*80)
    print("✅ 外部验证完成!")
    print("="*80)
    print(f"📊 总结:")
    print(f"   总样本数: {len(result_df)}")
    print(f"   预测为那非: {result_df['预测标签'].sum()}")
    print(f"   预测为非那非: {len(result_df) - result_df['预测标签'].sum()}")
    
    if 'label' in result_df.columns:
        if result_df['label'].nunique() > 1:  # 有真实标签
            accuracy = accuracy_score(result_df['label'], result_df['预测标签'])
            print(f"   📈 准确率: {accuracy:.4f}")
        else:
            print("   ⚠️  注意：所有样本的标签相同，无法计算准确率")
    
    print(f"\n📁 生成的文件:")
    print(f"   1. 预测结果: 外部验证1224_预测结果.xlsx")
    print(f"   2. ROC曲线图: external_validation_roc_curve.png")
    print(f"   3. 混淆矩阵图: external_validation_confusion_matrix.png")
    print("="*80)


if __name__ == "__main__":
    # 直接运行外部验证
    main_external_validation()